# 🪄 Chạy OpenAvatarChat miễn phí trên Colab GPU

Notebook này tự động:
1. Cài `uv` + Python 3.11 (đúng yêu cầu của dự án) và clone `OpenAvatarChat`.
2. Cài dependencies (torch, fastapi, các model handler...).
3. Vá file config: đổi `RtcClient` (WebRTC) → `WsClient` (WebSocket) để khớp với client React đã làm ở panel **My AI Avatar**, tắt SSL tự ký, tự điền LLM/giọng đọc bạn chọn.
4. **Vá ASR sang Groq Whisper** — model ASR mặc định (SenseVoiceSmall) chỉ hiểu Quan Thoại/Quảng Đông/Anh/Nhật/Hàn, KHÔNG hiểu tiếng Việt (nói tiếng Việt sẽ ra chữ Hán vô nghĩa). Notebook tự vá sang gọi Groq Whisper API (dùng chung key Groq ở Bước 1) để nhận diện tiếng Việt đúng.
5. Tải model ASR/TTS/LLM handler cần thiết.
6. **Dọn sạch server cũ** còn sót lại từ lần chạy trước (nếu có) để tránh tình trạng tunnel vô tình trỏ vào server cũ chưa được vá.
7. Chạy server ở cổng `8282` và mở **Cloudflare Quick Tunnel** miễn phí để lấy 1 URL `wss://...` public — dán URL đó vào ô "Server URL" trong panel.

> ⚠️ **Lưu ý quan trọng**: đây là môi trường Colab miễn phí — phiên sẽ tự ngắt sau vài giờ hoặc khi idle, và URL tunnel sẽ đổi mỗi lần chạy lại. Chỉ phù hợp để **test/demo**, không dùng cho sản phẩm chạy 24/7. GPU T4 miễn phí có thể chạy chậm hơn GPU cao cấp.
>
> ⚠️ **Mỗi khi chạy lại từ Bước 2** (phiên Colab mới, hoặc bấm "Run all" lại), các bước vá (4, 4b, 4c) đều cần chạy lại theo đúng thứ tự từ trên xuống — vì Bước 2 xoá và clone lại repo sạch, xoá luôn các bản vá cũ.

**Trước khi chạy**: vào `Runtime → Change runtime type → GPU (T4)` để bật GPU miễn phí.


In [ ]:
#@title Bước 0 — Kiểm tra GPU
!nvidia-smi || echo '⚠️ Chưa bật GPU. Vào Runtime > Change runtime type > GPU rồi chạy lại.'

In [ ]:
#@title Bước 1 — Cấu hình { display-mode: "form" }
GITHUB_REPO_URL = "https://github.com/HumanAIGC-Engineering/OpenAvatarChat.git" #@param {type:"string"}

#@markdown **LLM (bắt buộc)** — dùng API tương thích OpenAI. Mặc định đã đổi sang **Groq (miễn phí, không cần thẻ)**.
#@markdown Lấy API key miễn phí tại https://console.groq.com/keys rồi dán vào ô LLM_API_KEY bên dưới.
LLM_API_URL = "https://api.groq.com/openai/v1" #@param {type:"string"}
LLM_API_KEY = "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" #@param {type:"string"}
LLM_MODEL_NAME = "openai/gpt-oss-20b" #@param {type:"string"}
SYSTEM_PROMPT = "B\u1ea1n l\u00e0 m\u1ed9t tr\u1ee3 l\u00fd AI, tr\u1ea3 l\u1eddi ng\u1eafn g\u1ecdn 2-3 c\u00e2u b\u1eb1ng ti\u1ebfng Vi\u1ec7t." #@param {type:"string"}

#@markdown **Giọng đọc TTS (Edge TTS, miễn phí, không cần tải model nặng)** — ví dụ giọng nữ Việt: `vi-VN-HoaiMyNeural`, giọng nam: `vi-VN-NamMinhNeural`.
TTS_VOICE = "vi-VN-HoaiMyNeural" #@param {type:"string"}

#@markdown Tắt render video avatar (LiteAvatar) để tiết kiệm VRAM GPU free — client React hiện chỉ dùng audio + text nên vẫn hoạt động đầy đủ. Nếu server lỗi khi khởi động, thử bỏ tick rồi chạy lại từ Bước 4.
DISABLE_AVATAR_VIDEO = True #@param {type:"boolean"}

if not LLM_API_KEY:
    print('⚠️ Chưa điền LLM_API_KEY. Vào https://console.groq.com/keys tạo key miễn phí, dán vào ô trên rồi chạy lại ô này.')
else:
    print('Đã lưu cấu hình — dùng Groq:', LLM_API_URL, '| model:', LLM_MODEL_NAME)

In [ ]:
#@title Bước 2 — Clone repo + cài `uv`
%cd /content
!rm -rf OpenAvatarChat
!git clone --depth 1 {GITHUB_REPO_URL} OpenAvatarChat
%cd /content/OpenAvatarChat

# silero_vad là git submodule — clone --depth 1 không tự tải, phải init thủ công
# (nếu sau này bật thêm handler khác cũng là submodule, ví dụ LiteAvatar, cần init thêm dòng tương ứng)
!git submodule update --init src/handlers/vad/silerovad/silero_vad

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version

In [ ]:
#@title Bước 3 — Tạo venv Python 3.11 và cài dependencies (mất vài phút vì tải torch/CUDA)
!uv venv --python 3.11.7 .venv
!uv pip install --python .venv/bin/python --editable .

# QUAN TRỌNG: 'kích hoạt' venv cho cả session Colab (kể cả các lệnh gọi uv
# ngầm bên trong install.py ở Bước 5) — nếu không, uv sẽ tự rơi về Python
# hệ thống của Colab thay vì .venv/bin/python, gây lệch môi trường khi chạy demo.py.
import os
venv_path = os.path.abspath(".venv")
os.environ["VIRTUAL_ENV"] = venv_path
os.environ["PATH"] = os.path.join(venv_path, "bin") + ":" + os.environ["PATH"]
print("VIRTUAL_ENV =", os.environ["VIRTUAL_ENV"])

In [ ]:
#@title Bước 4 — Vá config: RtcClient → WsClient, tắt SSL tự ký, điền LLM/giọng đọc
import pathlib

src_path = pathlib.Path("config/chat_with_openai_compatible_edge_tts.yaml")
text = src_path.read_text(encoding="utf-8")

rtc_block = (
    "      RtcClient:\n"
    "        module: client/rtc_client/client_handler_rtc\n"
    "        # max time a session will last for\n"
    "        connection_ttl: 900\n"
)
ws_block = (
    "      WsClient:\n"
    "        module: client/ws_client/ws_client_handler\n"
    "        connection_ttl: 900\n"
)

if rtc_block not in text:
    raise SystemExit(
        "Không tìm thấy khối 'RtcClient' đúng như dự kiến — có thể repo gốc đã đổi cấu trúc.\n"
        "Mở config/chat_with_openai_compatible_edge_tts.yaml và tự đổi thủ công:\n"
        "  RtcClient: -> module: client/rtc_client/client_handler_rtc\n"
        "thành:\n"
        "  WsClient: -> module: client/ws_client/ws_client_handler"
    )
text = text.replace(rtc_block, ws_block)

# Tắt SSL tự ký để cloudflared xử lý HTTPS/WSS ở tầng ngoài (server chạy plain http/ws nội bộ)
text = text.replace('cert_file: "ssl_certs/localhost.crt"', 'cert_file: ""')
text = text.replace('cert_key: "ssl_certs/localhost.key"', 'cert_key: ""')

if DISABLE_AVATAR_VIDEO:
    text = text.replace(
        "      LiteAvatar:\n        module: avatar/liteavatar/avatar_handler_liteavatar",
        "      LiteAvatar:\n        enabled: False\n        module: avatar/liteavatar/avatar_handler_liteavatar",
    )

text = text.replace('model_name: "qwen-plus"', f'model_name: "{LLM_MODEL_NAME}"')
text = text.replace(
    'api_url: "https://dashscope.aliyuncs.com/compatible-mode/v1"',
    f'api_url: "{LLM_API_URL}"',
)
text = text.replace(
    'system_prompt: "\u8bf7\u4f60\u626e\u6f14\u4e00\u4e2a AI \u52a9\u624b\uff0c\u7528\u7b80\u77ed\u7684\u4e24\u4e09\u53e5\u5bf9\u8bdd\u6765\u56de\u7b54\u7528\u6237\u7684\u95ee\u9898\uff0c\u5e76\u5728\u5bf9\u8bdd\u5185\u5bb9\u4e2d\u52a0\u5165\u5408\u9002\u7684\u6807\u70b9\u7b26\u53f7\uff0c\u4e0d\u9700\u8981\u8ba8\u8bba\u6807\u70b9\u7b26\u53f7\u76f8\u5173\u7684\u5185\u5bb9"',
    f'system_prompt: "{SYSTEM_PROMPT}"',
)
text = text.replace('voice: "zh-CN-XiaoxiaoNeural"', f'voice: "{TTS_VOICE}"')

if LLM_API_KEY:
    os.environ["DASHSCOPE_API_KEY"] = LLM_API_KEY  # default env var the handler reads
    text = text.replace(
        "        api_url:",
        f'        api_key: "{LLM_API_KEY}"\n        api_url:',
    )

pathlib.Path("config/colab_config.yaml").write_text(text, encoding="utf-8")
print("Đã tạo config/colab_config.yaml:\n")
print(text)

In [ ]:
#@title Bước 4b — Vá install.py để ép dùng đúng .venv (khắc phục uv tự rơi về Python hệ thống)
import pathlib

install_path = pathlib.Path("install.py")
install_text = install_path.read_text(encoding="utf-8")

target = '["uv", "pip", "install"'
patched = '["uv", "pip", "install", "--python", ".venv/bin/python"'

count = install_text.count(target)
if count == 0:
    print("⚠️ Không tìm thấy chuỗi cần vá — có thể install.py đã đổi cấu trúc, kiểm tra thủ công.")
else:
    install_text = install_text.replace(target, patched)
    install_path.write_text(install_text, encoding="utf-8")
    print(f"Đã vá {count} lệnh 'uv pip install' bên trong install.py để luôn chỉ định --python .venv/bin/python")

In [ ]:
#@title Bước 4c — Vá ASR: SenseVoice (không hiểu tiếng Việt) → Groq Whisper (hiểu tiếng Việt)
import pathlib

asr_path = pathlib.Path("src/handlers/asr/sensevoice/asr_handler_sensevoice.py")
asr_text = asr_path.read_text(encoding="utf-8")

# Đây là dòng thật trong source SenseVoice handler của OpenAvatarChat — nơi nó gọi
# model FunASR cục bộ để nhận diện giọng nói. Model này (iic/SenseVoiceSmall) CHỈ
# hiểu Quan Thoại / Quảng Đông / Anh / Nhật / Hàn — không có tiếng Việt, nên khi bạn
# nói tiếng Việt nó ép âm thanh vào âm tiết Quảng Đông gần giống nhất -> ra chữ Hán
# vô nghĩa. Ta thay đúng dòng này bằng 1 lệnh gọi Groq Whisper (hiểu tiếng Việt tốt),
# và giữ nguyên mọi thứ khác (VAD, buffer, threading) vì phần đó đã chạy đúng.
anchor = "res = self.model.generate(input=output_audio, batch_size_s=10)"

if anchor not in asr_text:
    raise SystemExit(
        "Không tìm thấy dòng gọi model SenseVoice như dự kiến — có thể repo gốc đã đổi "
        "cấu trúc file src/handlers/asr/sensevoice/asr_handler_sensevoice.py.\n"
        "Mở file đó lên, tìm dòng có 'self.model.generate(' và báo lại nội dung dòng đó "
        "để chỉnh lại patch cho khớp."
    )

# Hàm gọi Groq Whisper, trả về đúng định dạng [{'text': ...}] mà code gốc bên dưới
# đang mong đợi (dòng kế tiếp re.sub(r"<\|.*?\|>", "", res[0]['text']) vẫn chạy bình
# thường vì text thường không chứa các tag đó).
groq_whisper_helper = '''
def _groq_whisper_transcribe(audio_data, sample_rate=16000, model="whisper-large-v3-turbo", language="vi"):
    """Transcribe qua Groq Whisper API — thay cho SenseVoice cục bộ để hỗ trợ tiếng Việt."""
    import requests, numpy as np, io, wave, os as _os
    api_key = _os.environ.get("GROQ_API_KEY") or _os.environ.get("DASHSCOPE_API_KEY")
    if not api_key:
        print("[groq_whisper] Thiếu GROQ_API_KEY trong biến môi trường.")
        return [{"text": ""}]
    try:
        arr = np.asarray(audio_data)
        if arr.dtype != np.int16:
            arr = np.clip(arr, -1.0, 1.0)
            arr = (arr * 32767.0).astype(np.int16)
        buf = io.BytesIO()
        with wave.open(buf, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            wf.writeframes(arr.tobytes())
        buf.seek(0)
        resp = requests.post(
            "https://api.groq.com/openai/v1/audio/transcriptions",
            headers={"Authorization": f"Bearer {api_key}"},
            files={"file": ("audio.wav", buf, "audio/wav")},
            data={"model": model, "language": language, "response_format": "json"},
            timeout=30,
        )
        resp.raise_for_status()
        text = resp.json().get("text", "").strip()
        print(f"[groq_whisper] transcript: {text!r}")
        return [{"text": text}]
    except Exception as e:
        print(f"[groq_whisper] transcribe error: {e}")
        return [{"text": ""}]


'''

# Chèn hàm helper vào ngay đầu file (an toàn — không phụ thuộc import sẵn có của file).
if "_groq_whisper_transcribe" not in asr_text:
    asr_text = groq_whisper_helper + asr_text

# Thay dòng gọi SenseVoice cục bộ bằng lệnh gọi Groq Whisper.
asr_text = asr_text.replace(anchor, "res = _groq_whisper_transcribe(output_audio, sample_rate=16000)")

asr_path.write_text(asr_text, encoding="utf-8")

# Groq Whisper dùng chung key với LLM đã điền ở Bước 1.
import os
if LLM_API_KEY:
    os.environ["GROQ_API_KEY"] = LLM_API_KEY

print("✅ Đã vá ASR handler: SenseVoice → Groq Whisper (language=\'vi\').")
print("   (Model SenseVoice vẫn được tải ở Bước 5 nhưng không còn được dùng để nhận diện — vô hại, chỉ tốn thêm ít thời gian tải.)")
print("   ⚠️ Nhắc nhớ: nếu bạn chạy lại Bước 2 (clone lại repo) ở 1 phiên sau, ô patch này PHẢI được chạy lại trước Bước 5/7.")


In [ ]:
#@title Bước 4d — Vá VAD: tránh bị AI cắt ngang khi đang nói giữa chừng
import pathlib

vad_path = pathlib.Path("config/colab_config.yaml")
vad_text = vad_path.read_text(encoding="utf-8")

# Mặc định của OpenAvatarChat: end_delay=5000 samples ở 16kHz ~= chỉ 312ms im lặng
# là VAD đã chốt "bạn nói xong" và đẩy câu (có thể còn dang dở) sang ASR/LLM ngay.
# Một khoảng ngừng tự nhiên giữa câu (hít thở, ngập ngừng) thường dài hơn thế,
# nên VAD hay cắt ngang. Ta tăng các ngưỡng liên quan lên để chịu được khoảng
# ngừng tự nhiên (~1s) mà vẫn phản hồi nhanh khi bạn thực sự dứt lời.
old_vad_block = (
    "      SileroVad:\n"
    "        module: vad/silerovad/vad_handler_silero\n"
    "        speaking_threshold: 0.5\n"
    "        start_delay: 2048\n"
    "        end_delay: 5000\n"
    "        buffer_look_back: 5000\n"
    "        speech_padding: 512\n"
)
new_vad_block = (
    "      SileroVad:\n"
    "        module: vad/silerovad/vad_handler_silero\n"
    "        speaking_threshold: 0.5\n"
    "        start_delay: 2048\n"
    "        end_delay: 16000\n"
    "        early_end_delay: 6000\n"
    "        early_end_repeat_delay: 4800\n"
    "        buffer_look_back: 5000\n"
    "        speech_padding: 512\n"
)

if old_vad_block not in vad_text:
    raise SystemExit(
        "Không tìm thấy khối 'SileroVad' đúng như dự kiến trong config/colab_config.yaml "
        "— có thể Bước 4 chưa chạy, hoặc repo gốc đã đổi cấu trúc.\n"
        "Mở file đó lên, tìm khối 'SileroVad:' và tự thêm/sửa thủ công 3 dòng:\n"
        "  end_delay: 16000\n"
        "  early_end_delay: 6000\n"
        "  early_end_repeat_delay: 4800"
    )

vad_text = vad_text.replace(old_vad_block, new_vad_block)
vad_path.write_text(vad_text, encoding="utf-8")
print("Đã tăng ngưỡng VAD trong config/colab_config.yaml:\n")
print(new_vad_block)
print(
    "Ghi chú: nếu vẫn thấy bị cắt ngang, có thể tăng end_delay lên 20000-24000 "
    "(1.25-1.5s). Nếu ngược lại thấy AI phản hồi chậm rõ rệt sau khi bạn dứt lời, "
    "giảm bớt xuống ~12000."
)


In [ ]:
#@title Bước 4e — Vá TTS: giữ dấu tiếng Việt khi đọc (Edge TTS)
import pathlib

tts_path = pathlib.Path("src/handlers/tts/edgetts/tts_handler_edgetts.py")
tts_text = tts_path.read_text(encoding="utf-8")

# Handler TTS gốc được viết cho tiếng Trung: filter_text() chỉ giữ lại chữ Latin
# không dấu, số, và chữ Hán (\u4e00-\u9fff) trước khi đưa văn bản cho Edge TTS đọc.
# Mọi ký tự có dấu tiếng Việt (ạ, ộ, ế, ữ, đ...) đều bị regex này xoá sạch, nên dù
# giọng đọc là vi-VN-HoaiMyNeural (nữ, tiếng Việt) vẫn nghe như đọc sai/lạ vì mất
# hết dấu và thanh điệu. Ta thêm dải Unicode tiếng Việt vào danh sách ký tự được giữ.
old_pattern = (
    '        pattern = r"[^a-zA-Z0-9\\u4e00-\\u9fff,.\\~!?，。！？ ]"  '
    '# 匹配不在范围内的字符\n'
)
new_pattern = (
    '        pattern = r"[^a-zA-Z0-9\\u00C0-\\u1EF9\\u4e00-\\u9fff,.\\~!?，。！？ ]"  '
    '# 匹配不在范围内的字符 (đã thêm \\u00C0-\\u1EF9 để giữ dấu tiếng Việt)\n'
)

if old_pattern not in tts_text:
    raise SystemExit(
        "Không tìm thấy dòng 'pattern = ...' đúng như dự kiến trong "
        "src/handlers/tts/edgetts/tts_handler_edgetts.py — có thể repo gốc đã đổi.\n"
        "Mở file đó, tìm hàm filter_text(), và tự thêm \\u00C0-\\u1EF9 vào regex "
        "(ngay trước \\u4e00-\\u9fff) để không bị xoá dấu tiếng Việt."
    )

tts_text = tts_text.replace(old_pattern, new_pattern)
tts_path.write_text(tts_text, encoding="utf-8")
print("Đã vá filter_text() trong tts_handler_edgetts.py — giờ dấu tiếng Việt sẽ được giữ lại.")
print("Lưu ý: chạy cell này SAU Bước 2 (đã clone repo) và TRƯỚC Bước 5 (tải model / chạy server).")


In [ ]:
#@title Bước 5 — Tải model (ASR SenseVoice...). Có thể mất vài phút tùy đường truyền.
!.venv/bin/python install.py --config config/colab_config.yaml

In [ ]:
#@title Bước 6 — Cài Cloudflare Tunnel (miễn phí, không cần tài khoản)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
#@title Bước 6b — Dọn sạch server cũ (nếu có) đang chiếm cổng 8282
import subprocess, time

# Kill mọi tiến trình python đang chạy src/demo.py từ (các) lần chạy trước còn sót lại
# trong VM Colab này. Nếu không dọn, Bước 7 bên dưới sẽ báo lỗi "address already in use"
# và server MỚI (đã có patch ASR) sẽ không khởi động được — trong khi tunnel Cloudflare
# vẫn vô tình trỏ vào server CŨ (chưa patch) khiến bạn tưởng patch không có tác dụng.
subprocess.run("pkill -9 -f 'src/demo.py' || true", shell=True)
subprocess.run("fuser -k 8282/tcp 2>/dev/null || true", shell=True)
time.sleep(3)

# Xác nhận cổng 8282 đã trống
check = subprocess.run("lsof -i:8282 || true", shell=True, capture_output=True, text=True)
if check.stdout.strip():
    print("⚠️ Cổng 8282 vẫn còn tiến trình đang giữ:\n" + check.stdout)
    print("   Nếu Bước 7 vẫn báo 'address already in use', vào Runtime → Restart session rồi chạy lại từ Bước 2.")
else:
    print("✅ Cổng 8282 đã trống — sẵn sàng khởi động server mới ở Bước 7.")


In [ ]:
#@title Bước 7 — Chạy server + mở tunnel, lấy URL wss:// để dán vào panel React
import re
import socket
import subprocess
import time

server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    [".venv/bin/python", "src/demo.py", "--config", "config/colab_config.yaml"],
    stdout=server_log, stderr=subprocess.STDOUT, cwd="/content/OpenAvatarChat",
)
print(f"Server PID: {server_proc.pid} — đang khởi động, chờ tới khi cổng 8282 sẵn sàng (tối đa 5 phút)...")

def port_is_open(host, port, timeout=1.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

MAX_WAIT_SECONDS = 300
waited = 0
server_ready = False
while waited < MAX_WAIT_SECONDS:
    if server_proc.poll() is not None:
        print(f"❌ Server đã thoát sớm (exit code {server_proc.returncode}) — xem server.log bên dưới:")
        print(open("/content/server.log").read()[-4000:])
        break
    if port_is_open("localhost", 8282):
        server_ready = True
        print(f"✅ Server đã mở cổng 8282 sau ~{waited}s.")
        break
    time.sleep(3)
    waited += 3
    if waited % 15 == 0:
        print(f"  ... vẫn đang chờ ({waited}s) — model ASR/TTS/LLM có thể cần thời gian tải/nạp GPU lần đầu.")

if not server_ready:
    if server_proc.poll() is None:
        print("⚠️ Hết thời gian chờ nhưng server vẫn đang chạy — có thể cần thêm thời gian. Xem server.log:")
        print(open("/content/server.log").read()[-4000:])
    raise SystemExit("Dừng lại: server chưa sẵn sàng, đừng mở tunnel để tránh nhầm lẫn URL chết.")

tunnel_log = open("/content/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8282"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log_text = open("/content/tunnel.log").read()
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break

if public_url:
    ws_url = public_url.replace("https://", "wss://")
    print("✅ Server + tunnel đã sẵn sàng!\n")
    print(f"Dán URL này vào ô 'Server URL' trong panel My AI Avatar:\n\n   {ws_url}\n")
else:
    print("⚠️ Chưa lấy được URL tunnel. Kiểm tra log bên dưới:")
    print(open("/content/tunnel.log").read())
    print("\n--- server.log ---")
    print(open("/content/server.log").read()[-3000:])

In [ ]:
#@title (Tuỳ chọn) Xem log server / tunnel trực tiếp để debug
print("----- server.log (300 dòng cuối) -----")
!tail -n 300 /content/server.log
print("\n----- tunnel.log -----")
!cat /content/tunnel.log

In [ ]:
#@title (Tuỳ chọn) Dừng server + tunnel
try:
    server_proc.terminate()
    tunnel_proc.terminate()
    print("Đã dừng server và tunnel.")
except NameError:
    print("Chưa có process nào đang chạy trong session này.")

## Ghi chú
- Mỗi lần Colab ngắt phiên (đóng tab, hết giờ, idle quá lâu), bạn phải chạy lại **toàn bộ notebook từ Bước 2** và URL tunnel ở Bước 7 sẽ **thay đổi** — nhớ cập nhật lại ô "Server URL" trong panel React.
- ASR mặc định của dự án gốc (SenseVoiceSmall) không hỗ trợ tiếng Việt — notebook này đã tự vá (Bước 4c) để dùng Groq Whisper thay thế, giúp nhận diện tiếng Việt chính xác. Nếu muốn quay lại SenseVoice gốc (ví dụ để test tiếng Trung/Anh/Nhật/Hàn), chỉ cần bỏ qua ô Bước 4c.
- Nếu dùng `LLM_API_URL` của DashScope (Qwen), cần API key từ Alibaba Cloud (biến `DASHSCOPE_API_KEY`). Nếu dùng OpenAI, điền `LLM_API_KEY` = OpenAI key, `LLM_API_URL` = `https://api.openai.com/v1`, `LLM_MODEL_NAME` = ví dụ `gpt-4o-mini`.
- Đây là cấu hình rút gọn (Edge TTS thay vì CosyVoice, tắt LiteAvatar) để vừa với GPU T4 free-tier. Muốn avatar video/motion data thật, cần renderer WebGL riêng của dự án (chưa có trong client React hiện tại) và GPU mạnh hơn.
